In [1]:
import sys

sys.path.append("../../..")
from setup_figs import sns, np, shuffle, plt, Patch, Line2D, pd, clean_names

In [ ]:
mds = clean_names(pd.read_parquet("0.parquet"))

# Throughout layers

In [ ]:
feature = "Relative clause type"
name = ""
bbox_to_anchor = (0.45, 0.94)
hue_order = mds[feature].sort_values().unique()

In [ ]:
feature = "(Object frequency x Embedded frequency)"
name = "_false_positive"
mds[feature] = mds[feature].apply(lambda x: x.replace(".5", "").replace(".0", ""))
hue_order = mds[feature].sort_values().unique()
bbox_to_anchor = (0.4, 0.94)

In [ ]:
g = sns.relplot(
    shuffle(mds),
    x="coord_1",
    y="coord_2",
    hue=feature,
    hue_order=hue_order,
    col="representations.layer",
    col_wrap=4,
    kind="scatter",
    height=4,
    facet_kws={
        "sharex": False,
        "sharey": False,
        "despine": False,
        "margin_titles": True,
    },
    edgecolor=None,
    linewidth=0.25,
    s=5,
    rasterized=True,
)
g.fig.subplots_adjust(wspace=0.05, hspace=0.05)
g.set_titles(col_template="Layer {col_name}")
g.set_axis_labels("", "")
for ax in g.axes.flat:
    title = ax.get_title()
    ax.set_title("")
    l, h = ax.get_xlim()
    ax.set_xlim(1.15 * l, 1.15 * h)
    l, h = ax.get_ylim()
    ax.set_ylim(1.15 * l, 1.15 * h)
    ax.text(0.02, 0.98, title, transform=ax.transAxes, ha="left", va="top")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
sns.move_legend(
    g, loc="lower center", bbox_to_anchor=bbox_to_anchor, ncol=4, markerscale=5
)
plt.savefig(f"../../../paper/figs/BERT_RC_mean/MDS/mds_layers{name}.pdf", dpi=100)
plt.show()

# Hierarchical clustering

In [ ]:
feature_1 = "Relative clause type"
feature_2 = "Attachment site"
feature_3 = "Subject number"
size_1 = max(len(feature_1), mds[feature_1].str.len().max())
hue = feature_1.ljust(size_1) + "    " + feature_2
mds[hue] = mds[feature_1].str.ljust(size_1 + 8) + "    " + mds[feature_2]
hue_order = sorted(mds[hue].unique())
tmp = shuffle(mds[mds["representations.layer"] == 6])

In [ ]:
fig = plt.figure(figsize=(10, 10))
values = tmp[feature_3].unique()
params = dict(
    x="coord_1",
    y="coord_2",
    style=feature_3,
    hue=hue,
    hue_order=hue_order,
    palette="tab20",
    markers={values[0]: "X", values[1]: "o"},
    s=30,
    linewidth=0.5,
    rasterized=True,
)
ax = sns.scatterplot(data=tmp[tmp[feature_3] == values[0]], edgecolor="grey", **params)
ax = sns.scatterplot(
    data=tmp[tmp[feature_3] == values[1]], edgecolor="white", ax=ax, **params
)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_xlabel("")
ax.set_ylabel("")

handles, labels = ax.get_legend_handles_labels()
n_hue = len(hue_order)
hue_handles = [Patch(facecolor=h.get_markerfacecolor()) for h in handles[1 : 1 + n_hue]]
handles = handles[0:1] + hue_handles + handles[1 + n_hue : 1 + n_hue + 2] + handles[-1:]
labels = labels[0:1] + hue_order + labels[1 + n_hue : 1 + n_hue + 2] + labels[-1:]
# Make the first label bold
leg = ax.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    markerscale=3,
)
plt.savefig("../../../paper/figs/BERT_RC_mean/MDS/hierarchical_mds.pdf")
plt.show()

## Walkthrough

In [ ]:
def build_walkthrough(mds, feature_1, value_1, feature_2, value_2, feature_3):
    level = mds[mds["representations.layer"] == 6].copy()
    level["hue"] = level[feature_1]
    hue_order = level[feature_1].unique().tolist()
    level["Level"] = 1
    walkthrough = [level.copy()]
    level["hue"] = level[feature_2]
    hue_order += level[feature_2].unique().tolist()
    level.loc[level[feature_1] == value_1, "hue"] = "NA"
    level["Level"] = 2
    walkthrough.append(level.copy())
    level["hue"] = level[feature_3]
    hue_order += level[feature_3].unique().tolist()
    level.loc[
        (level[feature_1] == value_1) + (level[feature_2] == value_2),
        "hue",
    ] = "NA"
    level["Level"] = 3
    walkthrough.append(level.copy())
    walkthrough = pd.concat(walkthrough, ignore_index=True)
    palette = sns.color_palette("tab10")
    palette = palette[: len(hue_order)] + [sns.color_palette("tab20")[15]]
    hue_order += ["NA"]

    return shuffle(walkthrough), hue_order, palette

In [ ]:
values_1 = mds[feature_1].unique()
values_2 = mds[feature_2].unique()

In [ ]:
walkthrough, hue_order, palette = build_walkthrough(
    mds, feature_1, values_1[1], feature_2, values_2[0], feature_3
)

In [ ]:
g = sns.relplot(
    walkthrough,
    kind="scatter",
    x="coord_1",
    y="coord_2",
    hue="hue",
    hue_order=hue_order,
    col="Level",
    s=5,
    height=6,
    aspect=0.8,
    facet_kws={
        "sharex": False,
        "sharey": False,
        "despine": False,
        "margin_titles": True,
    },
    edgecolor=None,
    linewidth=0.25,
    palette=palette,
    legend=False,
    rasterized=True,
)
g.set_axis_labels("", "")
g.set_titles(col_template="")
g.fig.subplots_adjust(wspace=0.05)
for ax in g.axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color=color,
        markerfacecolor=color,
        label=label,
        linewidth=0,
    )
    for color, label in zip(palette, hue_order)
    if label != "NA"
]
legend_handles.insert(0, Line2D([0], [0], linewidth=0, label=feature_1))
legend_handles.insert(3, Line2D([0], [0], linewidth=0, label=feature_2))
legend_handles.insert(6, Line2D([0], [0], linewidth=0, label=feature_3))
plt.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(-0.7, 0.975),
    ncols=3,
    frameon=False,
    columnspacing=3.2,
    markerscale=1.25,
)
plt.savefig(f"../../../paper/figs/BERT_RC_mean/MDS/hierarchical_mds_walkthrough.pdf")
plt.show()

### Walkthrough complete

In [ ]:
walkthroughs = []
walkthrough, hue_order, palette = build_walkthrough(
    mds, feature_1, values_1[1], feature_2, values_2[0], feature_3
)
walkthrough["Row"] = 0
walkthroughs.append(walkthrough)
walkthrough, _, _ = build_walkthrough(
    mds, feature_1, values_1[1], feature_2, values_2[1], feature_3
)
walkthrough["Row"] = 1
walkthroughs.append(walkthrough)
walkthrough, _, _ = build_walkthrough(
    mds, feature_1, values_1[0], feature_2, values_2[0], feature_3
)
walkthrough["Row"] = 2
walkthroughs.append(walkthrough)
walkthrough, _, _ = build_walkthrough(
    mds, feature_1, values_1[0], feature_2, values_2[1], feature_3
)
walkthrough["Row"] = 3
walkthroughs.append(walkthrough)
walkthroughs = pd.concat(walkthroughs)
walkthroughs = shuffle(walkthroughs)

In [ ]:
g = sns.relplot(
    walkthroughs,
    kind="scatter",
    x="coord_1",
    y="coord_2",
    hue="hue",
    hue_order=hue_order,
    col="Level",
    row="Row",
    s=5,
    height=5,
    aspect=1,
    facet_kws={
        "sharex": False,
        "sharey": False,
        "despine": False,
        "margin_titles": True,
    },
    edgecolor=None,
    linewidth=0.25,
    palette=palette,
    legend=False,
    rasterized=True,
)
g.set_axis_labels("", "")
g.set_titles(col_template="", row_template="")
g.fig.subplots_adjust(wspace=0.05, hspace=0.1)
for ax in g.axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

legend_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color, label=label)
    for color, label in zip(palette, hue_order)
    if label != "NA"
]
legend_handles.insert(0, Line2D([0], [0], color="w", label=feature_1))
legend_handles.insert(3, Line2D([0], [0], color="w", label=feature_2))
legend_handles.insert(6, Line2D([0], [0], color="w", label=feature_3))
plt.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(-0.675, 4.7),
    ncols=3,
    frameon=False,
    columnspacing=3.5,
    markerscale=1.25,
)
plt.savefig(
    f"../../../paper/figs/BERT_RC_mean/MDS/hierarchical_mds_walkthrough_full.pdf",
    bbox_inches="tight",
    dpi=100,
)
plt.show()